# Lektion 09: 🔗 BUILD 2 – Der RAG-Assistent mit Quellen



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Mittel |
| **Voraussetzungen** | Lektion 07–08 |
| **Du baust** | Die Build-2-Version deines Knowledge Assistant: semantische Suche + Chain + ehrliche Quellenangaben + strukturierte Ausgaben |

## 🎯 Nach dieser Lektion kannst du:

- Retriever-Ergebnisse **in eine Chain** einschleusen (RunnablePassthrough/RunnableLambda)

- Antworten mit **Quellen aus den Metadaten** liefern (nicht vom Modell zitieren lassen!)

- das ehrliche „Weiß ich nicht" bei Trefferlosigkeit garantieren

- strukturierte Ausgaben mit **Pydantic** erzwingen

- Build 1 vs. Build 2 sachlich vergleichen

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🔗 Willkommen zu BUILD 2!

Gleiche Anwendung, bessere Architektur. Der große Unterschied zu Build 1:

```text
Frage → Retriever (Bedeutungssuche!) → top-k Chunks + Metadaten
     → Prompt mit {context} → Modell → Parser → Antwort
     → Quellenliste separat aus den Metadaten
```

**Warum die Quellen NICHT vom Modell kommen:** Das Modell kann zitieren, was es will – wir nehmen die Wahrheit aus den **Metadaten** der gefundenen Chunks. Das ist Determinismus statt Hoffnung.

In [2]:
# Index einmal bauen (aus Lektion 08 – als wiederverwendbares Muster):
from lernakademie.offline_embeddings import get_embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def baue_retriever(k=3):
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    alle_chunks = []
    for datei in sorted((ROOT / "data").glob("*.md")):
        alle_chunks.extend(splitter.split_documents(
            TextLoader(str(datei), encoding="utf-8").load()))
    db = Chroma(collection_name="build2", embedding_function=get_embeddings())
    db.add_documents(alle_chunks)
    return db.as_retriever(search_kwargs={"k": k})

retriever = baue_retriever()
print("Retriever bereit.")

Retriever bereit.


/tmp/ipykernel_13613/1070553092.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 🧠 Konzept: Kontext in die Chain einschleusen

**Das Problem:** Die Chain erwartet `{{context}}` und `{{question}}` – aber der Retriever liefert Documents, kein Dictionary. Die Brücke: **`RunnablePassthrough().assign(context=RunnableLambda(...))`** – es reichert die Eingabe um das Context-Feld an, **ohne** die Frage zu verlieren.

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from lernakademie.llm_client import get_chat_model
from lernakademie.offline_embeddings import hat_bezug

PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Du beantwortest Fragen NUR auf Basis des gegebenen Kontexts. "
     "Wenn die Antwort nicht im Kontext steht, antworte exakt mit: "
     "'Das steht in den Dokumenten leider nicht.'"),
    ("human", "Kontext:\n{context}\n\nFrage: {question}"),
])

NICHT_GEFUNDEN = "Das steht in den Dokumenten leider nicht."

def baue_rag_chain(retriever):
    modell = get_chat_model(temperature=0.0)

    def _kontext_holen(eingabe):
        frage = eingabe["question"]
        if not hat_bezug(frage):        # offline: ehrlich statt beliebige Treffer
            return ""
        dokumente = retriever.invoke(frage)
        return "\n\n---\n\n".join(d.page_content for d in dokumente)

    def _relevante_docs(frage):
        """Nur Treffer mit echtem Bezug (offline: Aehnlichkeit > 0)."""
        from lernakademie.config import is_offline_mode
        dokumente = retriever.invoke(frage)
        if is_offline_mode():
            frage_vektor = get_embeddings().embed_query(frage)
            dokumente = [d for d in dokumente
                          if any(a * b for a, b in zip(
                              frage_vektor, get_embeddings().embed_query(d.page_content)))]
        return dokumente

    def _quellen(eingabe):
        frage = eingabe["question"]
        if not hat_bezug(frage):
            return []
        return sorted({Path(d.metadata["source"]).name
                       for d in _relevante_docs(frage)})

    chain = (
        RunnablePassthrough().assign(context=RunnableLambda(_kontext_holen))
        | PROMPT | modell | StrOutputParser()
    )
    return chain, _quellen

chain, quellen_fuer = baue_rag_chain(retriever)

## ✏️ Deine Aufgabe: Die Funktion `beantworte` (Build 2)

Du vollendest den Build-2-Assistenten: Antwort + Quellen in einem strukturierten Dictionary.

**Anforderungen:**

- Schreibe `beantworte(frage)`, das `{{"antwort": ..., "quellen": [...], "gefundene_stellen": int}}` zurückgibt.

- Bei leerem Kontext: `antwort=NICHT_GEFUNDEN`, `quellen=[]`, `gefundene_stellen=0` (kein Modell-Call nötig!).

- Nutze die fertige `chain` und `quellen_fuer` aus den Zellen oben.

- Teste die drei Standardfragen: Urlaubstage / Passwörter / Weihnachtsgeld (ehrlich leer!).

**Schreibe deinen Code in die folgende Zelle:**

In [4]:
def beantworte(frage):
    if not hat_bezug(frage):
        return {"antwort": NICHT_GEFUNDEN, "quellen": [], "gefundene_stellen": 0}
    antwort = chain.invoke({"question": frage})
    quellen = quellen_fuer({"question": frage})
    if not quellen:
        return {"antwort": NICHT_GEFUNDEN, "quellen": [], "gefundene_stellen": 0}
    return {"antwort": antwort, "quellen": quellen,
            "gefundene_stellen": len(quellen)}

for frage in ("Wie viele Urlaubstage gibt es?",
              "Was gilt fuer Passwoerter?",
              "Wie hoch ist das Weihnachtsgeld?"):
    ergebnis = beantworte(frage)
    print("FRAGE:", frage)
    print("  ->", ergebnis["antwort"][:100])
    print("  Quellen:", ergebnis["quellen"], "\n")

FRAGE: Wie viele Urlaubstage gibt es?
  -> [OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf 
  Quellen: ['mitarbeiter-handbuch.md'] 

FRAGE: Was gilt fuer Passwoerter?
  -> [OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf 
  Quellen: ['it-handbuch.md'] 

FRAGE: Wie hoch ist das Weihnachtsgeld?
  -> Das steht in den Dokumenten leider nicht.
  Quellen: [] 



**💡 Warum diese Loesung funktioniert:**

Der Early-Exit verhindert sinnlose Modell-Aufrufe (und im Offline-Modell ehrlich-beliebige Treffer). Quellen kommen aus Metadaten, nie aus der Modell-Antwort. Damit ist Build 2 komplett – vergleiche unten mit Build 1!

### ✅ Richtig, wenn:

- [ ] Die Urlaubsfrage liefert eine Antwort und `mitarbeiter-handbuch.md` in den Quellen.

- [ ] Die Weihnachtsgeld-Frage liefert das ehrliche NICHT_GEFUNDEN mit `quellen=[]`.

- [ ] Der Rückgabewert ist immer ein Dictionary mit den drei Schlüsseln.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [5]:
urlaub = beantworte("Wie viele Urlaubstage gibt es?")
assert "mitarbeiter-handbuch.md" in urlaub["quellen"], f"Quelle fehlt: {urlaub['quellen']}"
assert urlaub["gefundene_stellen"] >= 1
weihnachten = beantworte("Wie hoch ist das Weihnachtsgeld?")
assert weihnachten["quellen"] == [] and "leider nicht" in weihnachten["antwort"]
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Early-Exit: `if not hat_bezug(frage): return {...leer...}` ganz am Anfang (spart Modell-Calls!).
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Antwort holen: `chain.invoke({"question": frage})`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Quellen: `quellen_fuer({"question": frage})` liefert die sortierte Liste.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Warum ist „Antworte nur aus dem Kontext" Eindämmung, aber KEINE Garantie gegen Halluzination?

2. Welche Frage bleibt unbeantwortet, obwohl die Information im System steckt? (→ Verbindungsfragen – dein Einstieg in die Knowledge-Graph-Lektionen!)

## 🧠 Konzept: Strukturierte Ausgaben (Pydantic)

**WARUM:** Dein Assistent soll manchmal **Daten** liefern, nicht Fließtext: Thema, Stichpunkte, Zahlen. **Pydantic** erzwingt ein **Formular** – das Modell muss liefern, was dein Code erwartet (Validierung inklusive).

In [6]:
from pydantic import BaseModel, Field

class Kurzanalyse(BaseModel):
    thema: str = Field(description="Thema des Texts in wenigen Wortern")
    stichpunkte: list[str] = Field(description="3-5 Kernaussagen als Stichpunkte")
    schaetzwortzahl: int = Field(description="Geschaetzte Wortzahl")

analysator = get_chat_model(temperature=0.0).with_structured_output(Kurzanalyse)
ergebnis = analysator.invoke((ROOT / "data" / "firmenprofil.md").read_text(encoding="utf-8")[:1500])
print("Typ:", type(ergebnis).__name__)
print("Thema:", ergebnis.thema)
print("Stichpunkte:", ergebnis.stichpunkte[:3])

Typ: Kurzanalyse
Thema: [OFFLINE] thema (simuliert)
Stichpunkte: []


### 🔎 Was sehen wir hier?
Kein String, sondern ein **echtes Objekt** mit Attribut-Zugriff (`ergebnis.thema`). Die `description`-Texte sind Teil der Anweisung an das Modell!

## 📊 Der große Vergleich: Build 1 vs. Build 2

| Kriterium | 🧱 Build 1 | 🔗 Build 2 |
|---|---|---|
| Suche | Wort-Überlappung | semantisch (Embeddings) |
| "freie Tage"-Frage | ❌ schwach | ✅ findet Urlaubs-Abschnitt |
| Prompt | eigene String-Kleberei | `ChatPromptTemplate` (validiert!) |
| Antwort | Text aus `choices[0]...` | Chain + Parser + optional Pydantic |
| Quellen | eigene Logik | aus Metadaten des Retrievers |
| **Restproblem** | alles manuell, keine Synonyme | **keine Beziehungen** ("Wer arbeitet woran?") – → Knowledge Graph! |

Auch Build 2 existiert als Modul (`lernakademie/build2.py`) – gleiche Architektur wie in diesem Notebook:

In [7]:
from lernakademie.build2 import beantworte as b2
ergebnis = b2("Wie viele Urlaubstage gibt es?")
print("Antwort:", ergebnis["antwort"][:120])
print("Quellen:", ergebnis["quellen"])

Antwort: [OFFLINE] Simulierte Antwort (kein echtes Modell, keine Kosten). Ein echtes Modell wuerde jetzt auf folgenden Text reagi
Quellen: ['mitarbeiter-handbuch.md']


## ➡️ Weiter

**🕸️ Neue Welt – Lektion 10:** `notebooks/06_knowledge_graph/10_graphen_in_python.ipynb` – warum „Wer arbeitet woran?" keine Textfrage ist, sondern eine **Beziehungsfrage**.